# Milvus Vector Database Workshop
### Mastering Semantic Search with Milvus Lite

 This notebook is designed to run seamlessly on **Google Colab** using **Milvus Lite**.

**Workshop Agenda:**
1. **Environment Setup**: Installing Milvus Lite and dependencies.
2. **The Fundamentals**: Understanding Chunking, Embeddings, and Indexing.
3. **Data Preparation**: Different approaches to data cleaning and structuring.
4. **Initializing Milvus Collection**: Mastering schema parameters and pros/cons.
5. **Vectorization and Ingestion**: Batching and Streaming for large-scale data.
6. **Mastering Indexing**: Strategies for Milvus Lite and performance tuning.
7. **Semantic & Hybrid Search**: Finding the needle in the haystack.
8. **Data Retrieval**: How Milvus returns your original information.

### Link to Milvus documentation.
https://milvus.io/docs


## 1. Environment Setup
We use `pymilvus[milvus_lite]` which allows us to run Milvus directly inside this notebook without needing a Docker container or a separate server.

In [ ]:
# Install dependencies
!pip install -q pymilvus[milvus_lite] sentence-transformers pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 14.1 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd
import numpy as np
from pymilvus import MilvusClient, DataType
from sentence_transformers import SentenceTransformer

# Cleanup old database if it exists
if os.path.exists("milvus_demo.db"):
    os.remove("milvus_demo.db")


## 2. Deep Dive: The Theory

### A. Understanding Chunking
Before we can turn text into vectors, we must break it into manageable pieces. This is called **Chunking**.

| Chunking Type | Description | Pros | Cons |
| :--- | :--- | :--- | :--- |
| **Fixed-Size** | Splits at exactly X characters/tokens. | Simple, computationally cheap. | Often cuts sentences in half, losing context. |
| **Recursive** | Splits by characters (like paragraphs, then sentences). | Maintains semantic boundaries better. | Requires tuning separators. |
| **Semantic** | Splits where the "meaning" of the text changes. | Highest quality retrieval. | Very slow/expensive (requires LLM/Embeddings). |

### B. Different Embedding Models
Embeddings convert text into a list of numbers (vectors) where similar meanings are close together.

| Model | Origin | Dimension | Pros | Cons |
| :--- | :--- | :--- | :--- | :--- |
| **all-MiniLM-L6-v2** | HuggingFace | 384 | Extremely fast, low memory. | Lower accuracy on complex reasoning. |
| **all-mpnet-base-v2** | HuggingFace | 768 | High accuracy, standard for RAG. | Slower than MiniLM. |
| **text-embedding-3-small** | OpenAI | 1536 | SOTA performance, handles long text. | Paid API, data leaves your server. |

### C. How Hybrid Search Works
Hybrid search combines multiple search techniques to improve accuracy.
1. **Semantic Search (Dense Vectors)**: Finds conceptually similar items (e.g., searching for "happy" finds "joyful").
2. **Keyword Filtering (Scalar Filtering)**: Filters results based on hard attributes (e.g., `year > 2020` or `sentiment == 'positive'`).
3. **Reciprocal Rank Fusion (RRF)**: A method to combine scores from different search methods (like semantic + keyword) into a single ranked list.

## 3. Data Preparation: Different Approaches

Data prep is the foundation of high-quality search. Here are common approaches:

1.  **Cleaning**: Removing HTML tags, emojis, or excessive whitespace to reduce noise in embeddings.
2.  **Structuring**: Extracting metadata (dates, categories, ratings) during ingestion to enable scalar filtering later.
3.  **Augmentation**: Adding context to chunks (e.g., prepending the document title to every chunk) so the embedding model knows the "subject."

In [ ]:
# Load dataset
URL = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
df = pd.read_csv(URL).head(500) # Use 500 rows for speed

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


### Different Embedding Models
Embeddings convert text into a list of numbers (vectors) where similar meanings are close together.

| Model | Origin | Dimension | Pros | Cons |
| :--- | :--- | :--- | :--- | :--- |
| **all-MiniLM-L6-v2** | HuggingFace | 384 | Extremely fast, low memory. | Lower accuracy on complex reasoning. |
| **all-mpnet-base-v2** | HuggingFace | 768 | High accuracy, standard for RAG. | Slower than MiniLM. |
| **text-embedding-3-small** | OpenAI | 1536 | SOTA performance, handles long text. | Paid API, data leaves your server. |


### Embedding model token limits
The chunk size must be less than or equal to the model's token limit


In [ ]:
# Initialize Embedding Model
model = SentenceTransformer('all-MiniLM-L6-v2') #token limit 512
num_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {len(df)} rows and model with {num_params:,} parameters.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 500 rows and model with 22,713,216 parameters.


## 4. Initializing Milvus Collection: Parameters & Pros/Cons

A Collection is a container for your data, similar to a Table in SQL.

### Available Parameters in `create_collection`:
*   `dimension`: Number of elements in the vector (e.g., 384 for MiniLM).
*   `auto_id`: If `True`, Milvus automatically assigns a unique ID to each record.
*   `enable_dynamic_field`: If `True`, you can insert any metadata without pre-defining it in a schema (**Pros**: Extreme flexibility. **Cons**: No strict type checking).
*   `metric_type`: How distance is measured (e.g., `L2` for Euclidean, `COSINE` for angle similarity).
*   `consistency_level`: Controls when data becomes searchable after insertion (e.g., `Strong`, `Bounded`, `Session`, `Eventually`).

### Schema-based vs. Dynamic Schema
*   **Dynamic (Current approach)**: Just provide a dimension and start inserting. Great for prototyping.
*   **Explicit Schema**: Define every field and its type. Recommended for production to ensure data integrity.

In [ ]:
COLLECTION_NAME = "imdb_reviews"
client = MilvusClient("milvus_demo.db")

# Create collection
if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=384,
    auto_id=True,
    enable_dynamic_field=True,
    metric_type="COSINE"
)
print(f"Collection '{COLLECTION_NAME}' created with COSINE metric.")

Collection 'imdb_reviews' created with COSINE metric.


In [ ]:
# ### example of explicit schema
# from pymilvus import  DataType

# # 1. Initialize Client
# client = MilvusClient("movie_agent_strict.db")

# # 2. Create the Schema Object
# schema = client.create_schema(
#     auto_id=False,           # We will provide our own IDs
#     enable_dynamic_field=False # STRICT MODE: No extra fields allowed
# )

# # 3. Add Fields explicitly
# # Primary Key (Integer)
# schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)

# # Movie Title (String - up to 255 chars)
# schema.add_field(field_name="title", datatype=DataType.VARCHAR, max_length=255)

# # The Review Vector (384 Dimensions)
# schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dimension=384)

# # Sentiment (String)
# schema.add_field(field_name="sentiment", datatype=DataType.VARCHAR, max_length=10)

# # 4. Finalize the Collection
# client.create_collection(
#     collection_name="strict_movie_reviews",
#     schema=schema
# )

## 5. Vectorization and Ingestion: Streaming vs. Batching

When handling large files, you cannot load everything into memory.

1.  **Batch Ingestion**: Grouping records into batches (e.g., 100-1000) before calling `client.insert`. This reduces the number of network/disk calls.
2.  **Streaming Ingestion**: Using a generator to read a large CSV line by line, vectorizing on the fly, and inserting in small chunks.

Below is a demonstration of **Batch Ingestion**, which is the best balance of speed and memory usage.

### Understanding Chunking
Before we can turn text into vectors, we must break it into manageable pieces. This is called **Chunking**.

| Chunking Type | Description | Pros | Cons |
| :--- | :--- | :--- | :--- |
| **Fixed-Size** | Splits at exactly X characters/tokens. | Simple, computationally cheap. | Often cuts sentences in half, losing context. |
| **Recursive** | Splits by characters (like paragraphs, then sentences). | Maintains semantic boundaries better. | Requires tuning separators. |
| **Semantic** | Splits where the "meaning" of the text changes. | Highest quality retrieval. | Very slow/expensive (requires LLM/Embeddings). |

Below is a demonstration of **Fixed-size**


In [ ]:
batch_size = 100
data_to_insert = []

print("Starting Batch Ingestion...")
for i, row in df.iterrows():
    text = row['review'].replace("<br />", " ")
    vector = model.encode(text).tolist()

    data_to_insert.append({
        "vector": vector,
        "text": text[:500],
        "sentiment": row['sentiment']
    })

    # Every 100 records, insert the batch
    if len(data_to_insert) >= batch_size:
        client.insert(collection_name=COLLECTION_NAME, data=data_to_insert)
        data_to_insert = []
        print(f"Inserted batch ending at index {i}")


# 3. If there is anything left in the bucket after the loop, push it now
if data_to_insert:
    client.insert(collection_name=COLLECTION_NAME, data=data_to_insert)
    print(f"✅ Ingested final {len(data_to_insert)} leftover records.")

print("Ingestion complete.")

Starting Batch Ingestion...
Inserted batch ending at index 99
Inserted batch ending at index 199
Inserted batch ending at index 299
Inserted batch ending at index 399
Inserted batch ending at index 499
Ingestion complete.


## 6. Mastering Indexing for Milvus Lite

### What works with Milvus Lite?
Milvus Lite supports major indexing algorithms like `FLAT`, `IVF_FLAT`, and `HNSW`. Since Milvus Lite runs on your local CPU/Memory, **HNSW** is usually the best choice because it provides the best latency-to-recall ratio. **HNSW** does not work in milvus_lite

### Strategy based on Use Case:
| Use Case | Strategy | Index Type |
| :--- | :--- | :--- |
| **Small Dataset (< 5k)** | Brute force is fast enough. | `FLAT` |
| **Low Latency / Web App** | Fast retrieval via graphs. | `HNSW` (M=16, efConstruction=64) |
| **Memory Constrained** | Grouping vectors into buckets. | `IVF_FLAT` (nlist=1024) |
| **Billions of records** | Compression/Quantization. | `IVF_PQ` |



In [ ]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="vector",
    index_type="IVF_FLAT",
    metric_type="COSINE",
    params={"M": 16, "efConstruction": 64}
)

client.create_index(
    collection_name=COLLECTION_NAME,
    index_params=index_params
)
print("IVF_FLAT Index optimized for Milvus Lite created.")

IVF_FLAT Index optimized for Milvus Lite created.


## 7. Search vs. Hybrid Search: The Difference

1.  **Standard Semantic Search**: Finds the top results based ONLY on vector similarity. It might return a negative review if it uses similar descriptive language to your positive query.
2.  **Hybrid Search (Vector + Scalar)**: Finds results that are semantically similar AND match a metadata condition. This is far more powerful for real-world apps where you need to filter by user ID, date, or category.

### How Hybrid Search Works
Hybrid search combines multiple search techniques to improve accuracy.
1. **Semantic Search (Dense Vectors)**: Finds conceptually similar items (e.g., searching for "happy" finds "joyful").
2. **Keyword Filtering (Scalar Filtering)**: Filters results based on hard attributes (e.g., `year > 2020` or `sentiment == 'positive'`).
3. **Reciprocal Rank Fusion (RRF)**: A method to combine scores from different search methods (like semantic + keyword) into a single ranked list.

In [ ]:
query = "movie taking place in the nordic region"
query_vector = model.encode(query).tolist()

# HYBRID SEARCH EXECUTION
results = client.search(
    collection_name=COLLECTION_NAME,
    data=[query_vector],
    limit=3,
    filter="sentiment == 'positive'", # THE HYBRID FILTER
    output_fields=["text", "sentiment"] # DEFINING RETRIEVAL
)

print(f"\nResults for: '{query}' (Positive Only)")
for hit in results[0]:
    print(f"\n[Distance: {hit['distance']:.4f}] Sentiment: {hit['entity']['sentiment']}") # COSINE similarity, the distance score is a value between 0 and 1, where 1.0 is a perfect match
    print(f"Text Snippet: {hit['entity']['text']}...")


Results for: 'movie taking place in the nordic region' (Positive Only)

[Distance: 0.5113] Sentiment: positive
Text Snippet: I was very lucky to see this film as part of the Melbourne International Film Festival 2005 only a few days ago. I must admit that I am very partial to movies that focus on human relations and especially the ones which concentrate on the tragic side of life. I also love the majority of Scandinavian cinematic offerings, there is often a particular deep quality in the way the story unfolds and the characters are drawn. Character building in this film is extraordinary in its details and its depth. ...

[Distance: 0.4471] Sentiment: positive
Text Snippet: I saw this at the London Film Festival last night, apparently the shorter version. James McNally's summary of the content of the film is very good. Nossiter very deftly blends his investigation of the wine business into wider concerns about globalisation, homogenisation, the effect of the mass media, the power of c

## 8. Data Retrieval: How Original Data is Returned

By default, Milvus only returns the `id` and the `distance` (similarity score) of the match. To get your original text or metadata back, you must use the `output_fields` parameter in the `search` call.

### How it works:
1.  **Storage**: Milvus stores your metadata fields (like `text` or `sentiment`) in its own storage engine.
2.  **Request**: When you search, you specify `output_fields=["text", "rating"]`.
3.  **Hydration**: Milvus finds the matching IDs, fetches the associated metadata for those IDs, and "hydrates" the result object before sending it back to you.

**Pro Tip**: For very large metadata (like full books), it's often better to store only a `document_id` in Milvus and keep the actual text in a fast NoSQL database like MongoDB or Redis.